# 06 | Historical Feature Engineering
**NHS A&E Intelligent Decision Support System**

**Objective:** convert the verified M06 provider calendar (`docs/model_input_manifest.json`) into leakage-safe historical forecasting examples aligned with `docs/FORECAST_PROTOCOL.md` (`o = T - 2`, decision date = last calendar day before `T`).

All feature calculations are implemented in `src/models/feature_engineering.py` and verified by `tests/test_feature_engineering.py`. No train/test splits or models are fitted in this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/model_input_manifest.json").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the project repository before running this notebook.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.cohort_selection import load_accepted_m06
from src.models.feature_engineering import (
    ALL_FEATURE_COLUMNS,
    CORE_FEATURE_COLUMNS,
    OPTIONAL_FEATURE_COLUMNS,
    TARGET_COLUMN,
    build_feature_dataset,
    get_predictor_matrix,
    summarise_feature_dataset,
)

panel, manifest = load_accepted_m06(project_root)
features = build_feature_dataset(panel, verify_existing_eligibility=True)
summary = summarise_feature_dataset(features)

print("M06 run ID:", manifest["provenance"]["m06_run_id"])
print("M06 SHA-256:", manifest["dataset"]["sha256"])
display(pd.DataFrame([
    ("Eligible forecast origins", summary["eligible_origins"]),
    ("Distinct eligible providers", summary["distinct_providers"]),
    ("In-range origins (T <= 2026-09)", summary["in_range_origins"]),
    ("Rows with observed target labels", summary["target_observed_rows"]),
    ("Absent targets in range (dissolved trusts)", summary["absent_target_rows_in_range"]),
    ("Future forecast origins (T > 2026-09)", summary["future_forecast_rows"]),
    ("Rows with complete core features", summary["complete_core_feature_rows"]),
    ("Evaluable rows with complete features", summary["evaluable_with_complete_all_features"]),
], columns=["Metric", "Count"]))


## 1. Worked Provider Example
Inspect Manchester University NHS Foundation Trust (`R0A`) predicting **October 2025** (`target_period = 2025-10-01`) on decision date **`2025-09-30`**, using latest published report **`2025-08-01`** (`o = T - 2`).

In [ ]:
sample = features.loc[
    (features["org_code"] == "R0A") & (features["target_period"] == pd.Timestamp("2025-10-01"))
].iloc[0]
display(sample.to_frame(name="R0A (target 2025-10)"))

# Cross-check directly against the accepted M06 calendar
r0a_cal = panel.loc[panel["org_code"] == "R0A", ["period", "type1_attendances", "type1_emergency_admissions", "type1_over_4hrs"]].set_index("period")
check_months = ["2024-08-01", "2024-10-01", "2025-05-01", "2025-06-01", "2025-07-01", "2025-08-01", "2025-09-01", "2025-10-01"]
display(r0a_cal.loc[pd.to_datetime(check_months)])


## 2. Explicit Missing-Feature, Absent-Target, and Future-Origin Audit
Verify that initial `2022-12` origins retain explicit `NaN` for `type1_origin_lag_12`, absent targets from dissolved trusts remain `<NA>` rather than zero, and `2026-08`/`2026-09` future forecast origins are clearly distinguished.

In [ ]:
missing_df = pd.Series(summary["missing_counts_by_feature"], name="missing_rows").to_frame()
display(missing_df.loc[missing_df["missing_rows"] > 0])

absent_targets = features.loc[~features["is_future_forecast"] & ~features["target_observed"], ["org_code", "origin_period", "decision_date", "target_period", TARGET_COLUMN, "type1_origin"]]
print(f"Absent targets in range: {len(absent_targets)}")
display(absent_targets)

X = get_predictor_matrix(features, include_optional=True)
print("Predictor matrix shape:", X.shape, "| Target in X:", TARGET_COLUMN in X.columns)
